# 앙상블 전용 (CPU 런타임)

GPU 없이 **저장된 확률 파일만으로** 앙상블 점수 확인 + 제출 파일 생성.
학습이 돌고 있는 동안 새 코랩 창(런타임 유형: CPU)에서 쓴다.

- 멤버는 `ENS` 에 한 줄씩 (홀드아웃 H2000 확률, test 확률)
- 후보 조합은 `CANDS` 에 **미리 이유를 정해서** 몇 개만 적는다 (결과 보고 이리저리 고르면 과적합)
- 균등 평균만 쓴다

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import numpy as np, pandas as pd, torch
R = "/content/drive/MyDrive/ssafy_ai"
S, J = f"{R}/sunghyun", f"{R}/jeongyeon"

# H2000 정답과 test id 순서 (둘 다 TEMPLATE 순서와 같다)
y   = pd.read_csv(f"{S}/hold_pred_ens_H2000_0.9580.csv")["answer"].str.strip().str.lower().values
ids = pd.read_csv(f"{S}/sub_ens_equal_7b_zs_q3_8b_ft_q3_32b_ft_ocr_H2000_0.9585.csv")["id"]

def lp(p):
    x = torch.load(p, weights_only=False); x = x.get("probs", x) if isinstance(x, dict) else x
    return np.asarray(x)
acc = lambda P: (np.array(list("abcd"))[P.argmax(1)] == y).mean()

def tag(m, img, n, e=1): return f"{m}_choice_ce_shuf_img{img}_r16_n{n}_e{e}_H2000"
ENS = {   # 이름: (홀드아웃 H2000 확률, test 확률)
    "7b_zs":     (f"{S}/big2000_7b.pt",           f"{S}/test_zeroshot_7b_img768.pt"),
    "8b_ft768":  (f"{S}/valid_{tag('q3_8b',768,2000)}.pt",   f"{S}/test_{tag('q3_8b',768,2000)}_r0.pt"),
    "8b_ft1024": (f"{J}/valid_{tag('q3_8b',1024,4714)}.pt",  f"{J}/test_{tag('q3_8b',1024,4714)}_r0.pt"),
    "32b_ft":    (f"{S}/valid_{tag('q3_32b',768,2000)}.pt",  f"{S}/test_{tag('q3_32b',768,2000)}_r0.pt"),
    "ocr":       (f"{S}/valid_ocrmatch_H2000.pt",  f"{S}/test_ocrmatch.pt"),
}
CANDS = {
    "E6_현재최고": ["7b_zs", "8b_ft768", "8b_ft1024", "32b_ft", "ocr"],
    "E6_ocr제외": ["7b_zs", "8b_ft768", "8b_ft1024", "32b_ft"],
}

V = {k: lp(v) for k, (v, _) in ENS.items()}; T = {k: lp(t) for k, (_, t) in ENS.items()}
for k in ENS:
    assert V[k].shape == (2000, 4) and T[k].shape == (len(ids), 4), (k, V[k].shape, T[k].shape)
    print(f"{k:12s} {acc(V[k]):.4f}")

res = {}
for name, names in CANDS.items():
    res[name] = acc(sum(V[k] for k in names) / len(names))
    print(f"\n[{name}] {' + '.join(names)} → 균등 {res[name]:.4f}")
    for d in names:
        r = [k for k in names if k != d]
        print(f"   - {d:12s} 빼면 {acc(sum(V[k] for k in r)/len(r)):.4f}")

best = max(res, key=res.get); names = CANDS[best]
P = sum(T[k] for k in names) / len(names)
sub = f"sub_ens_equal_{'_'.join(names)}_H2000_{res[best]:.4f}.csv"
out = pd.DataFrame({"id": ids, "answer": np.array(list("abcd"))[P.argmax(1)]})
out.to_csv(f"{S}/{sub}", index=False)
print("\n선택:", best, "→", sub, out["answer"].value_counts(normalize=True).round(3).to_dict())